## **Module 6: Deferrable Sensors, TaskFlow API & BigQuery Operators**

In Module 4, we used a placeholder `BashOperator` to simulate checking whether upstream data was ready before launching our PySpark batch job. In production data pipelines on **Google Cloud Composer**, you should instead use **Sensors** and **BigQuery Data Quality Operators**.


### 1. Standard Sensors vs. Deferrable Sensors

* **Standard Sensors (`poke` mode):** Occupy an Airflow worker slot continuously while sleeping between checks. If dozens of DAGs wait on upstream files, worker slots can quickly be exhausted.
* **Deferrable Sensors (`deferrable=True`):** Offload the waiting phase to the lightweight **Airflow Triggerer** component using Python `asyncio`. The worker slot is immediately released and only re-allocated once the condition is met.


In [ ]:
# Wait asynchronously for an upstream GCS manifest file using Deferrable Mode
from airflow.providers.google.cloud.sensors.gcs import GCSObjectExistenceSensor
from airflow.providers.google.cloud.sensors.bigquery import BigQueryTableExistenceSensor

wait_for_manifest = GCSObjectExistenceSensor(
    task_id="wait_for_daily_manifest",
    bucket="{{ var.value.get('raw_landing_bucket', 'example-dataproc-workshop') }}",
    object="incoming/sales/{{ ds }}/_SUCCESS",
    deferrable=True,
    poke_interval=30,
    timeout=1800,
)


### 2. Combining `@task` (TaskFlow API) with Native BigQuery Operators

When transformations can be expressed directly in SQL, pushing compute down to **BigQuery** via `BigQueryInsertJobOperator` is faster and more cost-effective than spinning up a Spark cluster. You can also chain `BigQueryCheckOperator` to enforce data quality assertions before downstream consumers read the table.


In [ ]:
# Assert that the destination BigQuery partition contains at least 1 row
from airflow. providers.google.cloud.operators.bigquery import (
    BigQueryInsertJobOperator,
    BigQueryCheckOperator,
)

validate_output_not_empty = BigQueryCheckOperator(
    task_id="validate_output_not_empty",
    sql="""
        SELECT COUNT(1) > 0
        FROM `poc-example-ds.example_composer_workshop.daily_sales_summary`
        WHERE processing_date = DATE('{{ dag_run.conf.get("processing_date", ds) }}')
    """,
    use_legacy_sql=False,
    location="EU",
)


### 3. Complete Example

See [`dags/sensor_bq_etl_dag.py`](dags/sensor_bq_etl_dag.py) for the complete end-to-end DAG combining:
1. `GCSObjectExistenceSensor` & `BigQueryTableExistenceSensor` (`deferrable=True`)
2. `@task` metadata preparation
3. `BigQueryInsertJobOperator` partitioned & clustered aggregation
4. `BigQueryCheckOperator` post-load verification
